# Your First AI Agent -- From Chatbot to Agent

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Define what an AI agent is and how it differs from a basic chatbot.
2. Understand the **Perception -- Reasoning -- Action** loop that underpins every agent.
3. Set up a local LLM environment using **Ollama** and **LangChain 1.x**.
4. Build a minimal agent using the latest `create_agent` API.
5. Trace the full execution flow -- from user query to tool invocation to final response.

---

## Prerequisites

| Requirement | Minimum Version | Purpose |
|---|---|---|
| python | 3.11+ | Runtime |
| langchain | 1.0+ | Agent framework (latest API) |
| langchain-ollama | latest | Ollama integration |
| langgraph | latest | Graph-based agent runtime |
| python-dotenv | latest | Environment variable management |
| ollama | latest | Local LLM server |

...

> **Important:** This notebook uses the **LangChain 1.x** API. The older `create_react_agent` from
> `langgraph.prebuilt` still works but is no longer the recommended high-level entry point.
> The current best practice is to use `create_agent` from `langchain.agents`.

---

## 1. What is an AI Agent?

A **chatbot** receives a prompt and returns a completion. That is its entire capability -- it is a stateless input/output function.

An **AI agent** goes further. It can:

| Capability | Chatbot | Agent |
|---|---|---|
| Generate text | Yes | Yes |
| Use external tools | No | Yes |
| Make multi-step decisions | No | Yes |
| Observe tool results and adapt | No | Yes |

The key insight is that an agent treats the language model not as the final answer generator, but as a **reasoning engine** that decides *which actions to take* and *in what order*.

### Industry Definitions

Leading AI organizations have converged on similar definitions:

- **Anthropic:** Systems where LLMs dynamically direct their own processes and tool usage.
- **OpenAI:** Systems that independently accomplish tasks on behalf of users.
- **LangChain:** Systems using an LLM to decide the control flow of an application.

The common thread is **autonomy** -- the model decides what to do next, not the developer.

---

## 2. The Agent Loop: ReAct (Reasoning + Acting)

Every modern agent follows the **ReAct** pattern, which alternates between reasoning and acting:

```
User Query
    |
    v
+------------------+
|   REASONING      |  <-- LLM analyzes the query and decides: respond OR call a tool
+------------------+
    |
    v
+------------------+
|   ACTION         |  <-- Execute the tool call with the arguments the LLM chose
+------------------+
    |
    v
+------------------+
|   OBSERVATION    |  <-- Tool result is fed back to the LLM as a new message
+------------------+
    |
    v
(Loop back to REASONING until the LLM has enough info to respond)
    |
    v
Final Response
```

**Why this matters:**  
Traditional chatbots run exactly one forward pass. Agents run as many passes as the task requires. This is what makes them capable of solving complex, multi-step problems.

---

## 3. LangChain 1.x Agent Architecture

In LangChain 1.x, the recommended way to build agents is through the `create_agent` function. This function builds a **graph-based agent runtime** using LangGraph under the hood.

### How `create_agent` Works

```
create_agent(model, tools, system_prompt)
```

Internally, it creates a graph with two nodes:

1. **Model Node** -- Calls the LLM with the current messages list. If the LLM returns `tool_calls` in its response, execution moves to the Tools Node.
2. **Tools Node** -- Executes the requested tools and appends the results as `ToolMessage` objects. Execution then loops back to the Model Node.

This cycle repeats until the LLM returns a response with no `tool_calls`, at which point the graph returns the final message list.

### What Changed from the Old API?

| Old API (pre-1.x) | New API (1.x) |
|---|---|
| `from langgraph.prebuilt import create_react_agent` | `from langchain.agents import create_agent` |
| Lower-level, requires more configuration | High-level, batteries-included |
| `prompt` parameter (string) | `system_prompt` parameter (string or SystemMessage) |
| No middleware support | Built-in middleware system |
| No structured output | Native `response_format` support |

---

## 4. Environment Setup

### 4.1 Install Dependencies

Install the required packages.

---

In [ ]:
# Install required packages, if packages are not yet installed.

# uv add python-dotenv ollama langchain-ollama langchain

### 4.2 Verify Ollama is Running

Ollama must be running locally before we proceed. Open a terminal and run:

```bash
ollama serve
```

Then pull the model we will use:

```bash
ollama pull qwen3
```

> **Note:** You can substitute any Ollama-supported model that supports tool calling
> (e.g., `llama3.2`, `mistral`, `qwen3`). Not all models support tool calling --
> verify with `ollama show <model>` and check for tool support.

---

## 5. Configuration with Environment Variables

Hard-coding model names and parameters inside your source code is a common beginner mistake. It makes your code fragile and difficult to maintain across environments.

**Best practice:** Store all configuration in a `.env` file and load it at runtime.

Create a file named `.env` in the same directory as this notebook:

```
MODEL_NAME=minimax-m2.5:cloud
TEMPERATURE=0.7
```

---

In [ ]:
import os
from dotenv import load_dotenv

# Load variables from the .env file into the environment
load_dotenv()

# Read configuration with sensible defaults
MODEL_NAME = os.getenv("MODEL_NAME", "qwen3")
TEMPERATURE = float(os.getenv("TEMPERATURE", "0.7"))

print(f"Model       : {MODEL_NAME}")
print(f"Temperature : {TEMPERATURE}")

**What happened here:**

- `load_dotenv()` reads the `.env` file and injects each key-value pair into `os.environ`.
- `os.getenv("KEY", "default")` retrieves the value, falling back to the default if the key is absent.
- Casting `TEMPERATURE` to `float` ensures it is the correct type for the LLM constructor.

This pattern keeps secrets and configuration out of your codebase entirely.

---

## 6. Initialize the LLM

We use `ChatOllama` from `langchain_ollama` to connect to our local Ollama instance. This class wraps the Ollama HTTP API and exposes it through LangChain's standard chat model interface.

---

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model=MODEL_NAME,
    temperature=TEMPERATURE,
)

print(f"LLM initialized: {llm.model}")

In [ ]:
llm

**Key parameters:**

- **model** -- Must match a model you have already pulled with `ollama pull`.
- **temperature** -- Controls randomness. `0.0` is deterministic; `1.0` is highly varied. For tool-calling agents, values between `0.0` and `0.7` work best because the model needs to produce structured tool calls reliably.

---

## 7. Define a Tool

A **tool** is any Python function that the agent can invoke. For the LLM to use it correctly, the function must have:

1. A clear, descriptive **name** -- the LLM reads this to decide when to use it.
2. A **docstring** -- the LLM reads this to understand what the tool does.
3. **Type annotations** -- the LLM uses these to construct the correct arguments.

These three elements form the tool's **contract**. The LLM never sees the function body; it only sees the name, description, and parameter schema.

### Plain Function vs @tool Decorator

In LangChain 1.x, `create_agent` accepts **plain Python functions** directly as tools. You do not need the `@tool` decorator for basic use cases. The framework automatically converts the function into a tool using its name, docstring, and type hints.

The `@tool` decorator is useful when you need to customize the tool name, description, or argument schema beyond what the function signature provides.

---

In [ ]:
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"


# Verify the tool works independently before giving it to the agent
print(get_weather("Lahore"))

In [ ]:
# In production, this would call a real weather API.
# For learning purposes, return a static response.

**Why docstrings and type hints are critical for agents:**

When we register this function as a tool, LangChain extracts:
- The function name: `get_weather`
- The docstring: `"Get the current weather for a given city."`
- The parameter schema: `city: str`

This metadata is injected into the LLM prompt so it knows *what tools are available* and *how to call them*. If your docstring is vague or your type hints are missing, the LLM will make poor tool-calling decisions.

---

## 8. Create the Agent with `create_agent`

Now we combine the LLM and the tool into an **agent** using the LangChain 1.x API.

### Key Function Signature

```python
from langchain.agents import create_agent

agent = create_agent(
    model,                    # Required: LLM instance or model string
    tools=[...],              # Optional: list of tools (functions, BaseTool, or dict)
    system_prompt="...",      # Optional: system message for the LLM
)
```

`create_agent` returns a **compiled LangGraph** that handles the full ReAct loop automatically:

1. Send messages to the LLM.
2. If the LLM returns `tool_calls` -- execute the tools, append results, loop back to step 1.
3. If no `tool_calls` -- return the final response.

---

In [ ]:
from langchain.agents import create_agent

# Create the agent using the LangChain 1.x API
agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a helpful assistant. Use the available tools when needed.",
)

print("Agent created successfully.")
print(f"Type: {type(agent).__name__}")

In [ ]:
agent

**What `create_agent` does under the hood:**

1. Converts your plain Python functions into LangChain tool objects (extracting name, docstring, and type hints).
2. Binds the tools to the LLM so it knows how to format tool calls.
3. Builds a LangGraph with a **model node** and a **tools node** connected by conditional edges.
4. Compiles the graph into a runnable that you can invoke, stream, or batch.

You do not need to write any of this graph logic yourself. The framework handles it.

---

## 9. Run the Agent

We invoke the agent by passing a message in the standard chat format. The agent will:

1. Read the user message.
2. Reason that it needs weather data for Lahore.
3. Call `get_weather` with `city="Lahore"`.
4. Read the tool result.
5. Compose a natural language answer incorporating the tool data.

---

In [ ]:
# Invoke the agent with a user query
response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in Lahore?"}]}
)

# Extract and display the final response
print(response["messages"][-1].content)

In [ ]:
response

In [ ]:
response["messages"][-1]

**Understanding the output:**

The agent does not simply echo the tool's return value. It uses the LLM to compose a coherent, human-readable response that incorporates the data from the tool. This is a fundamental difference between calling a function directly and using an agent -- the agent contextualizes the raw data.

---

## 10. Streaming the Agent Output

For a better user experience, you can stream the agent's output instead of waiting for the entire response. The `stream` method yields updates as each node in the graph executes.

---

In [ ]:
# Stream the agent execution to see each step as it happens
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in Faisalabad?"}]},
    stream_mode="updates",
):
    # Each chunk is a dict with the node name as key
    for node_name, node_output in chunk.items():
        print(f"--- Node: {node_name} ---")
        if "messages" in node_output:
            for msg in node_output["messages"]:
                role = msg.__class__.__name__
                content = getattr(msg, "content", "")
                tool_calls = getattr(msg, "tool_calls", None)
                if tool_calls:
                    for tc in tool_calls:
                        print(f"  Tool Call: {tc['name']}({tc['args']})")
                if content:
                    print(f"  [{role}]: {content[:300]}")
    print()

**Why streaming matters:**

- In production applications, users expect immediate feedback rather than waiting several seconds for a complete response.
- Streaming also makes debugging easier because you can observe each step of the ReAct loop as it executes.
- The `stream_mode="updates"` parameter yields only the new state changes from each node, rather than the full accumulated state.

---

## 11. Inspect the Full Message History

To understand exactly what the agent did during execution, we can examine the complete sequence of messages.

---

In [ ]:
# Print every message in the execution trace
for i, msg in enumerate(response["messages"]):
    role = msg.__class__.__name__
    content = getattr(msg, "content", "")
    tool_calls = getattr(msg, "tool_calls", None)

    print(f"--- Message {i} [{role}] ---")
    if tool_calls:
        for tc in tool_calls:
            print(f"  Tool Call: {tc['name']}({tc['args']})")
    if content:
        print(f"  Content: {content[:300]}")
    print()

**Expected trace (simplified):**

```
Message 0 [HumanMessage]     -- "What is the weather in Lahore?"
Message 1 [AIMessage]        -- Tool Call: get_weather({"city": "Lahore"})
Message 2 [ToolMessage]      -- "It's always sunny in Lahore!"
Message 3 [AIMessage]        -- "It looks like the weather in Lahore is currently sunny! ☀️ It's described as always being sunny there. If you're planning any outdoor activities, you can expect nice weather! Is there anything else you'd like to know?"
```

This four-message sequence reveals the complete ReAct loop:
1. **Human** -- The user asks a question.
2. **AI (Reasoning)** -- The LLM decides to call a tool and specifies the arguments.
3. **Tool (Observation)** -- The tool executes and returns a result.
4. **AI (Final)** -- The LLM composes the final answer using the tool result.

---

## 12. Key Takeaways

1. **An agent is an LLM in a loop.** The model is called repeatedly, with tool results fed back as new context, until it produces a final answer.

2. **`create_agent` is the LangChain 1.x standard.** It replaces the older `create_react_agent` from `langgraph.prebuilt` as the recommended high-level entry point for building agents.

3. **Tools are plain Python functions.** The framework handles schema extraction, argument parsing, and execution. Your job is to write clear functions with good docstrings and type annotations.

4. **Configuration belongs in `.env` files.** Never hard-code model names, API keys, or tuning parameters.

5. **Always inspect the message trace.** Understanding the full execution flow is essential for debugging and improving agent behavior.

6. **Use streaming in production.** It provides better user experience and easier debugging.

---

## 13. API Quick Reference

```python
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

# Initialize model
llm = ChatOllama(model="qwen3", temperature=0.7)

# Define tools as plain functions (docstring + type hints required)
def my_tool(param: str) -> str:
    """Description of what this tool does."""
    return "result"

# Create agent
agent = create_agent(
    model=llm,                      # LLM instance or model string like "openai:gpt-5.3"
    tools=[my_tool],                # List of functions
    system_prompt="You are ...",    # System message
)

# Invoke (blocking)
result = agent.invoke({"messages": [{"role": "user", "content": "query"}]})

# Stream (real-time)
for chunk in agent.stream({"messages": [...]}, stream_mode="updates"):
    print(chunk)
```

---

## Exercises

Complete the following exercises to reinforce what you learned.

### Exercise 1: Add a Second Tool

Create a function called `get_time(city: str) -> str` that returns a simulated current time for a given city. Register it alongside `get_weather` and ask the agent: *"What is the weather and current time in Islamabad?"*

Observe the message trace. Does the agent call both tools? In what order?

### Exercise 2: Experiment with Temperature

Run the same query three times with `temperature=0.0` and three times with `temperature=1.0`. Compare the outputs. What do you notice about consistency and creativity?

### Exercise 3: Break the Docstring

Remove the docstring from `get_weather` and run the agent again. What happens? Why?

### Exercise 4: Use a Model String Instead of ChatOllama

Replace the `ChatOllama` instance with a model string (e.g., `"ollama:qwen3"`) passed directly to `create_agent`. Verify that the behavior is identical.

---

## What Comes Next

In **next notebook**, we will add **memory and persistence** to our agent so it can maintain context across multiple turns of conversation. You will learn:

- The difference between short-term (thread) and long-term (cross-thread) memory.
- How to use `checkpointer` for session-based memory.
- How to persist conversation history to a SQLite database.
- How to use `store` for cross-session knowledge.

---

*The School of Artificial Intelligence (SAI)*  